<a href="https://colab.research.google.com/github/nicopicomoco-sekihan-80/Antencoder-AntVLA/blob/main/experiment_of_v2.5s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python --version
!pip install -q gensim


Python 3.13.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 59.8 MB/s eta 0:00:00


In [2]:
import gensim
print(gensim.__version__)


4.4.0


In [5]:
!wget -q --show-progress \
  https://huggingface.co/fse/word2vec-google-news-300/resolve/main/GoogleNews-vectors-negative300.bin \
  -O GoogleNews-vectors-negative300.bin


In [6]:
!ls -lh



total 4.0K
-rw-r--r-- 1 root root    0 Sep 26 08:09 GoogleNews-vectors-negative300.bin
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [7]:
!pip install -q huggingface_hub


In [9]:
import gensim.downloader as api

print("Loading pretrained model (glove-wiki-gigaword-50)...")

model = api.load("glove-wiki-gigaword-50")

print("Model loaded successfully!")
print("Vector dimension:", model.vector_size)
print("Vocabulary size:", len(model))


Loading pretrained model (glove-wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded
Model loaded successfully!
Vector dimension: 50
Vocabulary size: 400000


In [10]:
words = [
    "robot",
    "object",
    "cup",
    "table",
    "pick",
    "place",
    "move",
    "left",
    "right",
]

for word in words:
    print(f"\n=== {word} ===")

    if word in model:
        for similar_word, score in model.most_similar(word, topn=5):
            print(f"{similar_word:15s} {score:.4f}")
    else:
        print("Not in vocabulary")



=== robot ===
robots          0.8780
robotic         0.8660
mouse           0.7269
humanoid        0.7007
simulator       0.6867

=== object ===
objects         0.8254
analogous       0.7574
precisely       0.7421
exists          0.7404
expression      0.7395

=== cup ===
champions       0.8477
championship    0.8157
tournament      0.8042
match           0.8033
finals          0.7987

=== table ===
tables          0.8177
place           0.7995
sit             0.7572
set             0.7330
open            0.7316

=== pick ===
picks           0.8855
picked          0.7905
picking         0.7783
spot            0.7746
go              0.7658

=== place ===
next            0.8695
time            0.8515
set             0.8455
rest            0.8385
where           0.8342

=== move ===
would           0.9008
could           0.8993
take            0.8992
to              0.8912
moves           0.8905

=== left ===
leaving         0.9067
back            0.8783
when            0.8685
after     

In [17]:
!pip install -U tensorflow-datasets


In [18]:
import tensorflow_datasets as tfds

ds = tfds.load(
    "bridge",
    split="train",
    shuffle_files=False
)

print(ds)


AttributeError: module 'tensorflow_datasets' has no attribute 'load'

In [19]:
import tensorflow_datasets as tfds

print(tfds)
print(tfds.__file__)
print(getattr(tfds, "__version__", "NO VERSION"))
print(hasattr(tfds, "load"))


<module 'tensorflow_datasets' from '/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py'>
/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py
NO VERSION
False


In [20]:
!pip show tensorflow-datasets


Name: tensorflow-datasets
Version: 4.9.10
Summary: tensorflow/datasets is a library of datasets ready to use with TensorFlow.
Home-page: https://github.com/tensorflow/datasets
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: absl-py, array_record, dm-tree, etils, immutabledict, numpy, promise, protobuf, psutil, pyarrow, requests, simple_parsing, tensorflow-metadata, termcolor, toml, tqdm, wrapt
Required-by: 


In [21]:
!pip install -q datasets


In [22]:
from datasets import load_dataset

ds = load_dataset(
    "MINT-SJTU/LA-33K",
    split="train"
)

print(ds)
print(ds.column_names)


README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

KeyboardInterrupt: 

In [23]:
!pip install -q huggingface_hub


In [24]:
from huggingface_hub import hf_hub_download

task_file = hf_hub_download(
    repo_id="MINT-SJTU/LA-33K",
    filename="meta/tasks.jsonl",
    repo_type="dataset"
)

print(task_file)


tasks.jsonl:   0%|          | 0.00/989k [00:00<?, ?B/s]

/root/.cache/huggingface/hub/datasets--MINT-SJTU--LA-33K/snapshots/3eeeb2c2fdabda173f174c6af16cfdfcd3a36f79/meta/tasks.jsonl


In [25]:
import json

tasks = []

with open(task_file, "r", encoding="utf-8") as f:
    for line in f:
        tasks.append(json.loads(line))

print("Number of records:", len(tasks))
print(tasks[0])


Number of records: 9532
{'task_index': 0, 'task': 'Move downward and forward to approach the object while holding nothing'}


In [26]:
instructions = [x["task"] for x in tasks]

print("Number of instructions:", len(instructions))

for x in instructions[:10]:
    print(x)


Number of instructions: 9532
Move downward and forward to approach the object while holding nothing
Close gripper to grasp the object
Move downward and slightly forward to approach the object while holding nothing
Move downward and slightly left to approach the bottom right corner of the object while holding nothing
Close gripper to grasp the object at the bottom right corner
Open gripper to release the object at the new position on the left
Move downward and slightly forward to approach the object on the object while holding nothing
Maintain grasp and lift the object vertically upward
Transport the object toward the object while rotating wrist to align orientation for insertion
Lower the object into the object while maintaining grasp


In [27]:
sentences = [
    instruction.lower().split()
    for instruction in instructions
]

print(sentences[:3])


[['move', 'downward', 'and', 'forward', 'to', 'approach', 'the', 'object', 'while', 'holding', 'nothing'], ['close', 'gripper', 'to', 'grasp', 'the', 'object'], ['move', 'downward', 'and', 'slightly', 'forward', 'to', 'approach', 'the', 'object', 'while', 'holding', 'nothing']]


In [28]:
from gensim.models import Word2Vec

vla_w2v = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=5,
    min_count=1,
    workers=1,
    sg=1,
    seed=42,
    epochs=20
)

print("Vector dimension:", vla_w2v.vector_size)
print("Vocabulary size:", len(vla_w2v.wv))


Vector dimension: 50
Vocabulary size: 1017


In [29]:
words = [
    "robot",
    "object",
    "cup",
    "table",
    "pick",
    "place",
    "move",
    "left",
    "right",
]

for word in words:
    print(f"\n=== {word} ===")

    if word in vla_w2v.wv:
        for w, score in vla_w2v.wv.most_similar(word, topn=5):
            print(f"{w:15s} {score:.4f}")
    else:
        print("Not in vocabulary")



=== robot ===
detergent       0.9043
enclosure       0.8929
else            0.8914
rigidly         0.8903
tab             0.8836

=== object ===
compartment     0.7360
banana          0.7346
cube            0.7252
cylinder        0.7215
marker          0.7036

=== cup ===
Not in vocabulary

=== table ===
Not in vocabulary

=== pick ===
Not in vocabulary

=== place ===
beside          0.7683
release         0.7164
slot            0.6924
outside         0.6820
second-last     0.6561

=== move ===
pieces          0.7293
scattered       0.6974
piano           0.6967
bookshelf       0.6908
keys            0.6890

=== left ===
right           0.9435
connected       0.6741
leftward        0.6336
spout           0.6114
interior        0.6044

=== right ===
left            0.9435
leftward        0.6808
connected       0.6200
pump            0.6020
spout           0.5902


In [30]:
common_words = set(model.key_to_index) & set(vla_w2v.wv.key_to_index)

print("Common vocabulary:", len(common_words))


Common vocabulary: 923


In [32]:
test_words = [
    "object",
    "place",
    "move",
    "left",
    "right",
    "robot",
    "release",
    "open",
    "close",
    "put",
]

for word in test_words:
    print(f"\n=== {word} ===")

    if word in common_words:
        print("GloVe:")
        for w, s in model.most_similar(word, topn=5):
            print(f"  {w:15s} {s:.3f}")

        print("VLA Word2Vec:")
        for w, s in vla_w2v.wv.most_similar(word, topn=5):
            print(f"  {w:15s} {s:.3f}")



=== object ===
GloVe:
  objects         0.825
  analogous       0.757
  precisely       0.742
  exists          0.740
  expression      0.740
VLA Word2Vec:
  compartment     0.736
  banana          0.735
  cube            0.725
  cylinder        0.722
  marker          0.704

=== place ===
GloVe:
  next            0.870
  time            0.851
  set             0.846
  rest            0.839
  where           0.834
VLA Word2Vec:
  beside          0.768
  release         0.716
  slot            0.692
  outside         0.682
  second-last     0.656

=== move ===
GloVe:
  would           0.901
  could           0.899
  take            0.899
  to              0.891
  moves           0.891
VLA Word2Vec:
  pieces          0.729
  scattered       0.697
  piano           0.697
  bookshelf       0.691
  keys            0.689

=== left ===
GloVe:
  leaving         0.907
  back            0.878
  when            0.869
  after           0.853
  out             0.851
VLA Word2Vec:
  right          

In [31]:
test_words = [
    "robot",
    "object",
    "place",
    "move",
    "left",
    "right",
]

for word in test_words:
    if word in common_words:
        print(f"{word}: OK")
    else:
        print(f"{word}: missing")


robot: OK
object: OK
place: OK
move: OK
left: OK
right: OK


In [39]:
import pandas as pd

url = (
    "https://huggingface.co/datasets/MINT-SJTU/LA-33K/"
    "resolve/main/data/chunk-000/episode_000000.parquet"
)

df = pd.read_parquet(
    url,
    columns=["action", "language_instruction"],
)

for i, row in df.head(5).iterrows():
    print(f"Frame {i}")
    print("  action:", row["action"])
    print("  language:", row["language_instruction"])


Frame 0
  action: [-0.22012979 -0.42091608 -0.13413987 -2.3519082  -0.1984668   2.2142367
  0.02765813  0.        ]
  language: 
Frame 1
  action: [-0.22080688 -0.42093784 -0.13538511 -2.35191    -0.19838211  2.2142518
  0.02759541  0.        ]
  language: 
Frame 2
  action: [-0.22102743 -0.42092547 -0.13638113 -2.3519175  -0.19826981  2.2142668
  0.02751344  0.        ]
  language: 
Frame 3
  action: [-0.22110277 -0.42090484 -0.13672633 -2.351919   -0.19823325  2.2142715
  0.02748699  0.        ]
  language: 
Frame 4
  action: [-0.22107121 -0.42092216 -0.13670333 -2.3519173  -0.19823349  2.2142754
  0.02748772  0.        ]
  language: 


In [46]:
import numpy as np

a = df["action"].iloc[0]

print("type:", type(a))
print("shape:", np.asarray(a).shape)
print("action:", a)
print("language:", df["language_instruction"].iloc[0])


type: <class 'numpy.ndarray'>
shape: (8,)
action: [ 0.08268997  0.19590029  0.03035069 -2.0961297  -0.14963004  2.4152267
  0.23789114  0.8843612 ]
language: Pick up the longer upright white container from the table and put it in the orange plastic bag


In [40]:
langs = df["language_instruction"].dropna().unique()

print("languages:", langs)


languages: ['']


In [45]:
import pandas as pd

for episode in range(10):
    url = (
        "https://huggingface.co/datasets/MINT-SJTU/LA-33K/"
        f"resolve/main/data/chunk-000/episode_{episode:06d}.parquet"
    )

    try:
        df = pd.read_parquet(
            url,
            columns=[
                "action",
                "language_instruction",
                "language_instruction_2",
                "language_instruction_3",
            ],
        )

        langs = set()

        for col in [
            "language_instruction",
            "language_instruction_2",
            "language_instruction_3",
        ]:
            langs.update(
                x for x in df[col].dropna()
                if str(x).strip()
            )

        print(f"episode {episode:06d}: {list(langs)[:3]}")

    except Exception as e:
        print(f"episode {episode:06d}: ERROR {e}")


episode 000000: []
episode 000001: []
episode 000002: ['Put the blue block inside the bowl', 'Put the blue block in the bowl', 'Put the blue block in the green bowl']
episode 000003: ['Pull the duvet to lift the bottom left duvet corner onto the top of the bed, pick the bottom left duvet corner and put it in the middle of the bed', 'Move the bottom right tip of the duvet to the left', 'fold the blanket']
episode 000004: ['Pull the duvet to lift the bottom left duvet corner onto the top of the bed, pick the bottom left duvet corner and put it in the middle of the bed', 'Move the bottom right tip of the duvet to the left', 'fold the blanket']
episode 000005: ['Pull the duvet to lift the bottom left duvet corner onto the top of the bed, pick the bottom left duvet corner and put it in the middle of the bed', 'Move the bottom right tip of the duvet to the left', 'fold the blanket']
episode 000006: ['Pick up the longer upright white container from the table and put it in the orange plastic b

In [ ]:
dataset = load_dataset(
    "MINT-SJTU/LA-33K",
    split="train",
    streaming=True
).select_columns(["task", "action"])

for i, row in enumerate(dataset):
    print("TASK :", row["task"])
    print("ACTION:", row["action"])
    print()

    if i >= 4:
        break
